# 05 — Final Speaker ↔ Transcript Alignment

Notebook นี้ใช้ข้อมูล final เท่านั้น และออกแบบมาแทน Notebook Speaker Alignment ตัวเก่า

Input:
- `dataset/mouth_crops_final/video001/mouth_metadata.csv`
- `dataset/transcripts/video001/transcript_with_frames.csv`

Output:
- `dataset/final_alignment/video001/final_speaker_transcript_alignment.csv`
- `dataset/final_alignment/video001/final_speaker_transcript_auto.csv`
- `dataset/final_alignment/video001/final_speaker_transcript_review.csv`
- `dataset/final_alignment/video001/segment_speaker_activity.csv`
- `dataset/final_alignment/video001/mouth_motion.csv`

หลักการ:
1. ใช้ mouth crops ของ `speaker_id 0/1`
2. คำนวณ lip motion ระหว่างเฟรม
3. สรุป activity ภายใน transcript segment
4. เปรียบเทียบ Speaker 0 กับ Speaker 1
5. ช่วงที่มั่นใจสูง → `AUTO`
6. ช่วงที่สูสี / ข้อมูลไม่พอ → `REVIEW` หรือ `NO_DATA`

## Cell 1 — Path / Config

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

MOUTH_METADATA_CSV = (
    PROJECT_ROOT / "dataset" / "mouth_crops_final"
    / VIDEO_ID / "mouth_metadata.csv"
)

TRANSCRIPT_CSV = (
    PROJECT_ROOT / "dataset" / "transcripts"
    / VIDEO_ID / "transcript_with_frames.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "dataset" / "final_alignment" / VIDEO_ID
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("MOUTH_METADATA_CSV:", MOUTH_METADATA_CSV)
print("TRANSCRIPT_CSV    :", TRANSCRIPT_CSV)
print("OUTPUT_ROOT       :", OUTPUT_ROOT)

if not MOUTH_METADATA_CSV.exists():
    raise FileNotFoundError(MOUTH_METADATA_CSV)

if not TRANSCRIPT_CSV.exists():
    raise FileNotFoundError(TRANSCRIPT_CSV)

## Cell 2 — Load Data

In [ ]:
import pandas as pd
import numpy as np

mouth_df = pd.read_csv(MOUTH_METADATA_CSV)
transcript_df = pd.read_csv(TRANSCRIPT_CSV)

if mouth_df["accepted"].dtype == object:
    mouth_df["accepted"] = (
        mouth_df["accepted"].astype(str).str.lower()
        .isin(["true", "1", "yes"])
    )

mouth_df = mouth_df[mouth_df["accepted"] == True].copy()
mouth_df["frame_index"] = mouth_df["frame_index"].astype(int)
mouth_df["speaker_id"] = mouth_df["speaker_id"].astype(int)

transcript_df["start_frame"] = transcript_df["start_frame"].astype(int)
transcript_df["end_frame"] = transcript_df["end_frame"].astype(int)

print("Accepted mouth rows :", len(mouth_df))
print("Speaker IDs         :", sorted(mouth_df["speaker_id"].unique().tolist()))
print("Transcript segments :", len(transcript_df))

display(mouth_df.head())
display(transcript_df.head())

## Cell 3 — Coverage QA

In [ ]:
max_mouth_frame = int(mouth_df["frame_index"].max())
max_transcript_frame = int(transcript_df["end_frame"].max())

coverage = (
    max_mouth_frame / max_transcript_frame
    if max_transcript_frame > 0 else 0.0
)

print("Max mouth frame     :", max_mouth_frame)
print("Max transcript frame:", max_transcript_frame)
print("Approx coverage     :", f"{coverage:.2%}")

if coverage < 0.90:
    print("\n[WARNING] mouth data ยังครอบคลุม transcript ไม่ถึง 90%")
else:
    print("\nCoverage OK")

## Cell 4 — Mouth Motion Functions

In [ ]:
import cv2
from pathlib import Path

def resolve_mouth_path(relative_path):
    if not isinstance(relative_path, str):
        return None
    relative_path = relative_path.strip()
    if not relative_path:
        return None

    p = Path(relative_path)
    return p if p.is_absolute() else PROJECT_ROOT / p


def read_mouth_gray(relative_path):
    p = resolve_mouth_path(relative_path)

    if p is None or not p.exists():
        return None

    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)

    if img is None:
        return None

    if img.shape != (96, 96):
        img = cv2.resize(
            img, (96, 96),
            interpolation=cv2.INTER_AREA
        )

    return img


def frame_difference(img_a, img_b):
    if img_a is None or img_b is None:
        return np.nan

    return float(
        cv2.absdiff(img_a, img_b).mean() / 255.0
    )

print("Motion functions ready")

## Cell 5 — Build Mouth Motion Table

In [ ]:
from tqdm.auto import tqdm

motion_rows = []

for speaker_id, group in mouth_df.groupby("speaker_id"):

    group = group.sort_values("frame_index").reset_index(drop=True)

    prev_img = None
    prev_frame = None

    for _, row in tqdm(
        group.iterrows(),
        total=len(group),
        desc=f"speaker {speaker_id}"
    ):
        current_frame = int(row["frame_index"])
        current_img = read_mouth_gray(row["mouth_file"])

        if (
            prev_img is not None
            and prev_frame is not None
            and current_frame - prev_frame <= 2
        ):
            motion = frame_difference(prev_img, current_img)
        else:
            motion = np.nan

        motion_rows.append({
            "speaker_id": int(speaker_id),
            "frame_index": current_frame,
            "motion": motion,
            "blur_score": row.get("blur_score", np.nan),
            "brightness": row.get("brightness", np.nan),
            "det_score": row.get("det_score", np.nan),
        })

        prev_img = current_img
        prev_frame = current_frame

motion_df = pd.DataFrame(motion_rows)

print("Motion rows:", len(motion_df))
display(motion_df.head(20))

## Cell 6 — Smooth Motion

In [ ]:
motion_df = motion_df.sort_values(
    ["speaker_id", "frame_index"]
).reset_index(drop=True)

motion_df["motion_smooth"] = (
    motion_df
    .groupby("speaker_id")["motion"]
    .transform(
        lambda s: s.rolling(
            window=5,
            center=True,
            min_periods=1
        ).median()
    )
)

display(motion_df.head(20))

## Cell 7 — Activity per Transcript Segment

In [ ]:
segment_activity_rows = []

for _, seg in tqdm(
    transcript_df.iterrows(),
    total=len(transcript_df),
    desc="transcript segments"
):
    segment_id = int(seg["segment_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])
    expected_frames = max(1, end_frame - start_frame + 1)

    matched = motion_df[
        (motion_df["frame_index"] >= start_frame)
        & (motion_df["frame_index"] <= end_frame)
    ]

    for speaker_id in [0, 1]:
        speaker_data = matched[
            matched["speaker_id"] == speaker_id
        ]

        valid_motion = speaker_data["motion_smooth"].dropna()
        observed_frames = int(
            speaker_data["frame_index"].nunique()
        )

        segment_activity_rows.append({
            "segment_id": segment_id,
            "speaker_id": speaker_id,
            "mean_motion": (
                float(valid_motion.mean())
                if len(valid_motion) else np.nan
            ),
            "median_motion": (
                float(valid_motion.median())
                if len(valid_motion) else np.nan
            ),
            "peak_motion": (
                float(valid_motion.quantile(0.90))
                if len(valid_motion) else np.nan
            ),
            "observed_frames": observed_frames,
            "coverage": min(
                1.0, observed_frames / expected_frames
            ),
        })

activity_df = pd.DataFrame(segment_activity_rows)

display(activity_df.head(20))

## Cell 8 — Speaker Activity Score

In [ ]:
activity_df["mean_motion"] = activity_df["mean_motion"].fillna(0.0)
activity_df["peak_motion"] = activity_df["peak_motion"].fillna(0.0)
activity_df["coverage"] = activity_df["coverage"].fillna(0.0)

activity_df["speaker_score"] = (
    0.55 * activity_df["mean_motion"]
    + 0.30 * activity_df["peak_motion"]
    + 0.15 * activity_df["coverage"]
)

display(
    activity_df.sort_values(
        ["segment_id", "speaker_score"],
        ascending=[True, False]
    ).head(30)
)

## Cell 9 — Final Speaker Assignment

In [ ]:
MIN_COVERAGE = 0.50
MIN_MEAN_MOTION = 0.008
MIN_RELATIVE_MARGIN = 0.18

alignment_rows = []

for _, seg in transcript_df.iterrows():

    segment_id = int(seg["segment_id"])

    candidates = activity_df[
        activity_df["segment_id"] == segment_id
    ].sort_values("speaker_score", ascending=False)

    if len(candidates) == 0:
        continue

    top = candidates.iloc[0]
    second = candidates.iloc[1] if len(candidates) > 1 else None

    top_score = float(top["speaker_score"])
    second_score = (
        float(second["speaker_score"])
        if second is not None else 0.0
    )

    margin = (
        (top_score - second_score)
        / max(top_score, 1e-8)
    )

    top_coverage = float(top["coverage"])
    top_mean_motion = float(top["mean_motion"])

    if top_coverage < MIN_COVERAGE:
        status = "NO_DATA"
    elif top_mean_motion < MIN_MEAN_MOTION:
        status = "REVIEW"
    elif margin < MIN_RELATIVE_MARGIN:
        status = "REVIEW"
    else:
        status = "AUTO"

    alignment_rows.append({
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "start_time": float(seg["start_time"]),
        "end_time": float(seg["end_time"]),
        "start_frame": int(seg["start_frame"]),
        "end_frame": int(seg["end_frame"]),
        "text": str(seg["text"]),
        "speaker_id": (
            int(top["speaker_id"])
            if status != "NO_DATA"
            else np.nan
        ),
        "speaker_score": round(top_score, 6),
        "second_score": round(second_score, 6),
        "relative_margin": round(float(margin), 6),
        "coverage": round(top_coverage, 6),
        "mean_motion": round(top_mean_motion, 6),
        "status": status,
    })

alignment_df = pd.DataFrame(alignment_rows)

display(alignment_df.head(30))

## Cell 10 — Summary

In [ ]:
print("Total segments:", len(alignment_df))
print()
print(alignment_df["status"].value_counts(dropna=False))

print("\nAUTO speaker counts:")
print(
    alignment_df[
        alignment_df["status"] == "AUTO"
    ]["speaker_id"].value_counts()
)

auto_rate = (
    (alignment_df["status"] == "AUTO").mean()
)

print(
    "\nAUTO segment rate:",
    f"{auto_rate:.2%}"
)

## Cell 11 — Export Final Alignment

In [ ]:
ALL_CSV = OUTPUT_ROOT / "final_speaker_transcript_alignment.csv"
AUTO_CSV = OUTPUT_ROOT / "final_speaker_transcript_auto.csv"
REVIEW_CSV = OUTPUT_ROOT / "final_speaker_transcript_review.csv"
ACTIVITY_CSV = OUTPUT_ROOT / "segment_speaker_activity.csv"
MOTION_CSV = OUTPUT_ROOT / "mouth_motion.csv"

alignment_df.to_csv(
    ALL_CSV,
    index=False,
    encoding="utf-8-sig"
)

alignment_df[
    alignment_df["status"] == "AUTO"
].to_csv(
    AUTO_CSV,
    index=False,
    encoding="utf-8-sig"
)

alignment_df[
    alignment_df["status"] != "AUTO"
].to_csv(
    REVIEW_CSV,
    index=False,
    encoding="utf-8-sig"
)

activity_df.to_csv(
    ACTIVITY_CSV,
    index=False,
    encoding="utf-8-sig"
)

motion_df.to_csv(
    MOTION_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(" -", ALL_CSV)
print(" -", AUTO_CSV)
print(" -", REVIEW_CSV)
print(" -", ACTIVITY_CSV)
print(" -", MOTION_CSV)

## Cell 12 — QA: REVIEW

In [ ]:
review_df = alignment_df[
    alignment_df["status"] != "AUTO"
].copy()

display(
    review_df[
        [
            "segment_id",
            "start_time",
            "end_time",
            "text",
            "speaker_id",
            "speaker_score",
            "second_score",
            "relative_margin",
            "coverage",
            "mean_motion",
            "status",
        ]
    ].head(100)
)

## Cell 13 — QA: AUTO Samples

In [ ]:
auto_df = alignment_df[
    alignment_df["status"] == "AUTO"
].copy()

if len(auto_df) > 0:
    sample_auto = auto_df.sample(
        n=min(30, len(auto_df)),
        random_state=42
    ).sort_values("start_time")

    display(
        sample_auto[
            [
                "segment_id",
                "start_time",
                "end_time",
                "text",
                "speaker_id",
                "relative_margin",
                "coverage",
                "mean_motion",
            ]
        ]
    )
else:
    print("ไม่มี AUTO segment")

# ขั้นต่อไป — 06 Final Labeled Lip Clip Builder

เมื่อ alignment ผ่าน ให้ใช้:
- `final_speaker_transcript_auto.csv`
- `mouth_metadata.csv`

เพื่อสร้าง labeled clips จริง แล้วค่อยทำ manifest / train / val / test